# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore, process, and visualize data from the **FAIR^2** dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/latest/) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) and is accessible via the following URL:

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Install `mlcroissant` if not yet installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Examine available record sets (tables), fields, and their `@id`s.

Record sets define the main tables or collections of records in the dataset. For each record set, we list its fields (columns), along with their unique Croissant `@id` and their meaning.

In [ ]:
# List available record sets (tables)
record_sets = list(metadata.record_sets)
print(f"Number of record sets found: {len(record_sets)}\n")
for i, record_set in enumerate(record_sets):
    print(f"{i + 1}. Record Set name: '{record_set.name}' | @id: '{record_set.id}'")
    if hasattr(record_set, 'description') and record_set.description:
        print(f"   Description: {record_set.description}")
    print("   Fields (columns):")
    for field in record_set.fields:
        print(f"      - {field.name} (id: {field.id}, type: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames. Each record set should be referenced by its Croissant `@id`, and columns/fields likewise by their `@id`.

In [ ]:
# Prepare to load all record sets into DataFrames, using their @ids
dataframes = {}
record_set_ids = [r.id for r in metadata.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} records for record set {record_set_id}")

# For demonstration, display the columns of the first record set
main_record_set_id = record_set_ids[0]
print(f"\nColumns for the main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records by value, normalize numeric columns, and group by a categorical field. All field accesses are by `@id`.

Let's select a numeric field (such as protein abundance, molecular weight, or similar) and a group/category field (if available), using their Croissant `@id`s.

**Adjust the field IDs as appropriate to your dataset.**

In [ ]:
# Replace with actual field IDs from the data overview above!
# For illustration, we will guess likely IDs for abundance and group fields.

df = dataframes[main_record_set_id]

# Show column names with their likely meanings
print("Available columns:", df.columns.tolist())

# TRY to select a likely numeric field (by guessing common protein data field names)
possible_numeric_ids = [col for col in df.columns if 'abundance' in col.lower() or 'mw' in col.lower() or 'weight' in col.lower() or 'count' in col.lower() or df[col].dtype in ['float64', 'int64']]
if possible_numeric_ids:
    numeric_field_id = possible_numeric_ids[0]
else:
    # Fallback: just use the first column
    numeric_field_id = df.columns[0]

# TRY to select a group field
possible_group_ids = [col for col in df.columns if 'sample' in col.lower() or 'type' in col.lower() or 'category' in col.lower()]
if possible_group_ids:
    group_field_id = possible_group_ids[0]
else:
    group_field_id = None  # not available

print(f"Numeric field candidate: {numeric_field_id}")
if group_field_id:
    print(f"Group (categorical) field candidate: {group_field_id}\n")

# Remove NaNs before filtering
df_clean = df.copy()
df_clean = df_clean[pd.to_numeric(df_clean[numeric_field_id], errors='coerce').notnull()]
df_clean[numeric_field_id] = pd.to_numeric(df_clean[numeric_field_id], errors='coerce')

# Simple filter: keep only records where the numeric field is greater than the median
threshold = df_clean[numeric_field_id].median()
filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > median ({threshold}):\n")
display(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# (Optional) Group by a categorical field, if available
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped.head())

## 5. Visualization
Visualize distribution of the numeric field and (if available) its grouping by category.

**Feel free to further customize this cell.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Plot the distribution of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id} (after filtering)")
plt.show()

# If a grouping is available, plot grouped means
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    order = filtered_df[group_field_id].value_counts().index
    sns.barplot(data=filtered_df, x=group_field_id, y=numeric_field_id, ci=None, order=order)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset via its FAIR^2 Croissant schema. We identified record sets using their `@id`, extracted fields, performed simple EDA, and visualized a key numeric attribute of the proteins. Further analysis could involve domain-specific statistics or downstream machine learning workflows.

**Remember:** always reference dataset entities and columns by their `@id` when using Croissant and `mlcroissant`.